# Problem 20: Multi-Step Tool Chain

**Difficulty:** Medium | **Framework:** LangChain

**Categories:** tool-calling, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/multi-step-tool-chain).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- All three tools (search, extract_entities, summarize) must be called in sequence
- The agent must not skip any step or jump directly to summarization
- Each tool's output must feed into the next tool as input
- No changes to the individual tool implementations


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def search_articles(query: str) -> str:
    """Search for articles on a topic."""
    return f"Found 3 articles about {query}: Article A discusses market trends, Article B covers recent innovations, Article C analyzes industry impact."

@tool
def extract_entities(text: str) -> str:
    """Extract key entities (people, companies, topics) from text."""
    return "Entities: [TechCorp, AI industry, John Smith, market growth, 2024 forecast]"

@tool
def summarize(text: str) -> str:
    """Summarize the given text into a concise brief."""
    return "Summary: The AI industry is experiencing rapid growth driven by TechCorp and similar companies, with analysts like John Smith forecasting continued expansion through 2024."

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant with access to search, entity extraction, and summarization tools."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# BUG: The agent has all 3 tools but often skips steps — it jumps straight to summarize
# or calls tools out of order. It should always follow: search -> extract_entities -> summarize
tools = [search_articles, extract_entities, summarize]
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools)

result = executor.invoke({"input": "Research the latest trends in AI and give me a brief"})
print(result["output"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Update the system prompt to explicitly instruct the agent to follow a step-by-step workflow: first search, then extract, then summarize
2. Consider enforcing ordering by making each tool check that the previous step's output exists before proceeding
3. Use few-shot examples in the prompt showing the correct three-step sequence


## 7. Evaluation Criteria

Check your solution against these criteria:

- All three tools are called
- Tools called in correct order
- Outputs are chained between steps
